# RAG Retriever — VN30 Financial Knowledge Base

**Mục đích notebook này:**  
Xây dựng và kiểm thử RAG pipeline — từ loading dữ liệu đến augmented prompt cho LLM.

**Kiến trúc tích hợp với FastAPI pipeline:**

```
POST /analyze
    │
    ├─ ratio_engine.compute  →  ratios (dict)
    ├─ scoring.score         →  score  (dict)
    └─ ai_analysis.analyze
           │
           ├─ RAG/rag_retriever.py  ← retrieve context từ pgvector
           └─ LLM API               ← nhận ratios + retrieved context
```

**Nguyên tắc thiết kế:**
- AI **chỉ nhận ratios đã tính sẵn** + data từ DB — không nhận raw BCTC
- RAG bổ sung **domain knowledge** (benchmark ngành, PDF báo cáo, dữ liệu lịch sử)
- Vector store: **pgvector** (cùng PostgreSQL đã có trong stack)
- Embedding và LLM: **chưa xác định** — cấu hình qua `RAGConfig`

**RAG pipeline:**
```
0. Dependencies + Environment
1. Data loading          ← text/PDF (phi cấu trúc) + DB (có cấu trúc)
2. Chunking              ← optimize chunk_size / overlap
3. Embedding → pgvector  ← optimize embedding model ($)
4. Retrieval + reranking ← optimize top_k / search_type
5. Prompt engineering    ← optimize prompt ($)
6. Evaluation
```

**Web visualize chunk:** https://www.chunkviz.com/

---
## 0. Dependencies + Environment

In [ ]:
# Chạy một lần để cài, sau đó comment lại
# %pip install langchain langchain-community langchain-postgres langchain-openai \
#              pypdf python-dotenv psycopg2-binary sqlalchemy

In [ ]:
import os
import sys
import json
from pathlib import Path
from dotenv import load_dotenv

# Trỏ về project root để import các module nội bộ
PROJECT_ROOT = Path(".").resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env")

# Kiểm tra env cần thiết
DATABASE_URL = os.environ["DATABASE_URL"]
print("DATABASE_URL:", DATABASE_URL.split("@")[-1])  # ẩn credentials

# API key tuỳ embedding provider đang dùng
# OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
# COHERE_API_KEY = os.environ["COHERE_API_KEY"]

In [ ]:
# Load config — tất cả tham số pipeline ở đây
from RAG.config import RAGConfig

cfg = RAGConfig()

# --- Điều chỉnh tham số tại đây trước khi chạy ---
# cfg.chunking.chunk_size = 800
# cfg.chunking.chunk_overlap = 150
# cfg.embedding.provider = "openai"
# cfg.embedding.model_name = "text-embedding-3-small"
# cfg.retrieval.top_k = 6
# --------------------------------------------------

print(json.dumps(cfg.summary(), indent=2))

## 1. Data loading

Nguồn dữ liệu: Có cấu trúc (txt/md/pdf) + Phi cấu trúc + DB

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document

kb_dir = cfg.loading.knowledge_base_dir

# --- 1a. Text / Markdown ---
text_loader = DirectoryLoader(
    kb_dir,
    glob=cfg.loading.text_glob,
    loader_cls=TextLoader,
    loader_kwargs={"encoding": cfg.loading.text_encoding},
    show_progress=True,
    use_multithreading=True,
)
text_docs = text_loader.load()
print(f"Text/MD docs: {len(text_docs)}")

pdf_loader = DirectoryLoader(
    kb_dir,
    glob=cfg.loading.pdf_glob,
    loader_cls=PyPDFLoader,
    show_progress=True,
    use_multithreading=True,
)
pdf_docs = pdf_loader.load()
print(f"PDF docs (pages): {len(pdf_docs)}")

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, UnstructuredFileLoader

# Load tất cả file văn bản/định dạng hỗn hợp qua Unstructured
untext_loader = DirectoryLoader(
    kb_dir,
    glob=cfg.loading.text_glob,       
    loader_cls=UnstructuredFileLoader,
    loader_kwargs={
        "mode": "elements",             # Phân tách rõ Title, NarrativeText, ListItem...
        "strategy": "fast",             # "fast" cho văn bản thuần, dùng "hi_res" nếu file có layout phức tạp
    },
    show_progress=True,
    use_multithreading=True,
)

untext_docs = untext_loader.load()
print(f"Unstructured Text/Doc docs: {len(untext_docs)}")

In [ ]:
# --- 1c. Dữ liệu có cấu trúc từ PostgreSQL ---
# Chuyển mỗi row (ticker + year + ratios + score) thành Document để chunk
# cùng pipeline với unstructured docs.

from sqlalchemy import create_engine, text as sql_text

engine = create_engine(DATABASE_URL)

DB_QUERY = """
    SELECT
    #     Đoạn SQL
"""

db_docs = []
with engine.connect() as conn:
    rows = conn.execute(sql_text(DB_QUERY)).mappings().all()

for row in rows:
    # Format thành text có cấu trúc để embedding hiểu được ngữ nghĩa
    content = (
        # merge các ô thành câu bằng func
            )
    db_docs.append(Document(
        page_content=content,
        metadata={
            # VD: "source": "db", "ticker": row["ticker"], "year": row["year"]
            },
    ))

print(f"DB docs (rows): {len(db_docs)}")

# Gộp tất cả nguồn
all_documents = text_docs + pdf_docs + untext_docs + db_docs
print(f"Total documents: {len(all_documents)}")

## 2. Chunking strategies *(optimize)*

Dùng `RecursiveCharacterTextSplitter`.  
Tham số chỉnh trong `cfg.chunking`.

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

c = cfg.chunking
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=c.chunk_size,
    chunk_overlap=c.chunk_overlap,
    add_start_index=c.add_start_index,
    strip_whitespace=c.strip_whitespace,
    keep_separator=c.keep_separator,
    separators=c.separators,
    is_separator_regex=c.is_separator_regex,
)

# DB docs đã nhỏ gọn — không cần split thêm, chỉ split text
unstructured_docs = text_docs + pdf_docs + untext_docs
chunks = text_splitter.split_documents(unstructured_docs) + db_docs

print(f"Unstructured chunks: {len(text_splitter.split_documents(unstructured_docs))}")
print(f"DB docs (no split): {len(db_docs)}")
print(f"Total chunks: {len(chunks)}")

# Kiểm tra chunk mẫu
if chunks:
    sample = chunks[0]
    print(f"\n--- Sample chunk (source={sample.metadata.get('source')}) ---")
    print(sample.page_content[:400])

---
## 3. Embedding → pgvector *(optimize: $ for model)*

Embedding model chọn qua `cfg.embedding.provider`.  
Swap provider bằng cách uncomment block tương ứng.

In [ ]:
from langchain_core.embeddings import Embeddings

e = cfg.embedding

def build_embedding_model(cfg_emb) -> Embeddings:
    provider = cfg_emb.provider.lower()

    if provider == "openai":
        from langchain_openai import OpenAIEmbeddings
        return OpenAIEmbeddings(
            model=cfg_emb.model_name,
            dimensions=cfg_emb.dimensions,
        )

    if provider == "google":
        from langchain_google_genai import GoogleGenerativeAIEmbeddings
        return GoogleGenerativeAIEmbeddings(
            model=cfg_emb.model_name,
            google_api_key=os.environ["GEMINI_API_KEY"],
        )

    if provider == "cohere":
        from langchain_cohere import CohereEmbeddings
        return CohereEmbeddings(
            model=cfg_emb.model_name,
            cohere_api_key=os.environ["COHERE_API_KEY"],
        )

    if provider == "huggingface":
        from langchain_community.embeddings import HuggingFaceEmbeddings
        return HuggingFaceEmbeddings(model_name=cfg_emb.model_name)

    raise ValueError(f"Unknown embedding provider: {provider!r}")


embedding_model = build_embedding_model(e)
print(f"Embedding: {e.provider} / {e.model_name} (dim={e.dimensions})")

In [ ]:
from langchain_postgres import PGVector

vs = cfg.vector_store

# Build vector store và upsert toàn bộ chunks
# pre_delete_collection=True: xoá collection cũ trước khi rebuild
vectorstore = PGVector.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name=vs.collection_name,
    connection=vs.connection_string,
    distance_strategy=vs.distance_strategy,
    pre_delete_collection=vs.pre_delete_collection,
)

print(f"Vector store built: collection='{vs.collection_name}', "
      f"strategy={vs.distance_strategy}")

## 4. Retrieval + reranking *(optimize)*

Tham số chỉnh trong `cfg.retrieval`.  
Reranker optional — bật bằng `cfg.retrieval.reranker_model`.

In [ ]:
r = cfg.retrieval

retriever = vectorstore.as_retriever(
    search_type=r.search_type,
    search_kwargs={
        "k": r.top_k,
        "fetch_k": r.fetch_k,         # dùng khi search_type="mmr"
        "lambda_mult": r.mmr_lambda,  # dùng khi search_type="mmr"
        # "score_threshold": r.score_threshold,  # dùng khi search_type="similarity_score_threshold"
    },
)

# Kiểm thử retrieval với query tài chính thực tế
test_query = "current_ratio=1.20, roe=0.08, debt_to_equity=1.50 — ngưỡng đánh giá VN30"
retrieved_docs = retriever.invoke(test_query)

print(f"Retrieved {len(retrieved_docs)} chunks (search_type={r.search_type})\n")
for i, doc in enumerate(retrieved_docs, 1):
    src = doc.metadata.get("source", "unknown")
    ticker = doc.metadata.get("ticker", "")
    year = doc.metadata.get("year", "")
    label = f"{src}" + (f" | {ticker} {year}" if ticker else "")
    print(f"[{i}] {label}")
    print(doc.page_content[:250])
    print("-" * 60)

In [ ]:
# --- Optional reranking ---
# Bỏ comment khi đã chọn reranker_model trong cfg.retrieval

# if r.reranker_model:
#     from langchain.retrievers import ContextualCompressionRetriever
#     from langchain_cohere import CohereRerank  # hoặc CrossEncoderReranker
#
#     compressor = CohereRerank(
#         model=r.reranker_model,
#         top_n=r.top_k,
#     )
#     retriever = ContextualCompressionRetriever(
#         base_compressor=compressor,
#         base_retriever=retriever,
#     )
#     retrieved_docs = retriever.invoke(test_query)
#     print(f"After reranking: {len(retrieved_docs)} chunks")

## 5. Prompt engineering *(optimize: $ for LLM)*

LLM chưa xác định — cell này build prompt cuối cùng và
gọi LLM thông qua interface LangChain (`BaseChatModel`) để dễ swap.

Tham số prompt chỉnh trong `cfg.prompt`.

In [ ]:
from ratio_engine.compute import StatementInput, compute_all_ratios
from scoring.score import compute_financial_health_score

# Dữ liệu mẫu
sample_statement = StatementInput(
    revenue=10_000, net_profit=800,
    total_assets=50_000, total_liabilities=30_000, total_equity=20_000,
    current_assets=8_000, current_liabilities=6_500, cash_from_operations=1_200,
)
ratios = compute_all_ratios(sample_statement)
score  = compute_financial_health_score(ratios)

print("Ratios:", json.dumps(ratios, indent=2))
print("\nScore:",  json.dumps(score,  indent=2))

In [ ]:
# Build RAG context từ retrieved docs
p = cfg.prompt

def build_context_block(docs, separator: str, max_tokens: int) -> str:
    """Nối các chunk lại, cắt bớt nếu vượt max_tokens (ước tính 4 char/token)."""
    max_chars = max_tokens * 4
    sections = []
    total = 0
    for doc in docs:
        src = Path(doc.metadata.get("source", "unknown")).name
        block = f"[Nguồn: {src}]\n{doc.page_content.strip()}"
        if total + len(block) > max_chars:
            break
        sections.append(block)
        total += len(block)
    return separator.join(sections)


rag_context = build_context_block(
    retrieved_docs,
    separator=p.chunk_separator,
    max_tokens=p.max_context_tokens,
)

print(f"Context block: {len(rag_context)} chars (~{len(rag_context)//4} tokens)")
print(rag_context[:600])

In [ ]:
SYSTEM_PROMPT = """Bạn là trợ lý phân tích tài chính. Bạn sẽ nhận vào:
1. Các chỉ số tài chính ĐÃ ĐƯỢC TÍNH SẴN (không phải báo cáo tài chính thô).
2. Ngưỡng tham chiếu + dữ liệu lịch sử từ knowledge base VN30.

Nhiệm vụ:
1. Phân tích xu hướng theo 5 nhóm: Liquidity, Profitability, Leverage, Efficiency, Cash flow.
2. Đưa ra dự đoán CÓ THỂ KIỂM CHỨNG cho năm tiếp theo:
   - trend: "improve" | "stable" | "decline"
   - risk:  "low" | "moderate" | "high"
   - profitability: "improve" | "stable" | "decline"
3. Giải thích ngắn gọn, đối chiếu với ngưỡng tham chiếu và lịch sử ngành.

CHỈ trả về JSON đúng schema, không thêm text khác:
{"trend_prediction": "...", "risk_prediction": "...", "profitability_prediction": "...", "explanation": "..."}
"""

full_prompt = (
    SYSTEM_PROMPT
    + f"\n\n{p.rag_section_header}:\n" + rag_context
    + f"\n\n{p.data_section_header}:\n"
    + json.dumps({"2024": ratios}, ensure_ascii=False, indent=2)
)

print(f"Full prompt length: {len(full_prompt)} chars")
print("\n--- Preview (first 800 chars) ---")
print(full_prompt[:800])

In [ ]:
# Gọi LLM qua LangChain interface — swap provider bằng cách thay dòng import
# Uncomment block tương ứng với LLM đang dùng:

# --- OpenAI ---
# from langchain_openai import ChatOpenAI
# llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# --- Google Gemini ---
# from langchain_google_genai import ChatGoogleGenerativeAI
# llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)

# --- Anthropic Claude ---
# from langchain_anthropic import ChatAnthropic
# llm = ChatAnthropic(model="claude-3-5-haiku-latest", temperature=0)

# --- Local (Ollama) ---
# from langchain_community.chat_models import ChatOllama
# llm = ChatOllama(model="llama3.2")

# Gọi sau khi chọn LLM ở trên:
# from langchain_core.messages import HumanMessage
# response = llm.invoke([HumanMessage(content=full_prompt)])
# result = json.loads(response.content)
# result["_prompt_version"] = p.version
# result["_config"] = cfg.summary()
# print(json.dumps(result, ensure_ascii=False, indent=2))

---
## 6. Evaluation

Log config vào `evaluation_results` để track qua các lần thử nghiệm.

In [ ]:
# Xem tóm tắt config của run này — paste vào evaluation_results.notes
import json
print(json.dumps(cfg.summary(), ensure_ascii=False, indent=2))

In [ ]:
# --- Load lại vector store (không rebuild) ---
# Dùng khi chỉ cần query, không cần upsert lại

# from langchain_postgres import PGVector
# vectorstore = PGVector(
#     embeddings=embedding_model,
#     collection_name=cfg.vector_store.collection_name,
#     connection=cfg.vector_store.connection_string,
#     distance_strategy=cfg.vector_store.distance_strategy,
# )